# ESG News Risk Forecaster: Colab Results Walkthrough

This notebook is the grader-facing demonstration for the DSC 148 project. It is designed to run in Google Colab from the GitHub repository outputs, without the 23 GB raw FNSPID CSV and without local Parquet intermediates.

**Research question:** Can ESG-sensitive financial news representations improve short-horizon stock-risk forecasting compared with market-only baselines?

**Main answer:** price-history features are a strong baseline. The best selected tuned configuration is **price + ClimateBERT**, which adds modest incremental value for both 21-trading-day realized volatility regression and high-volatility classification.

## How to Use This Notebook in Colab

Run the cells from top to bottom. If the notebook is opened directly from GitHub in Colab, the setup cell below clones the repository into `/content/esg-news-risk-forecaster`. If it is run locally inside the repository, it uses the local files.

The notebook intentionally reads only generated tables and figures from:

- `outputs/tables/`
- `outputs/figures/`
- `report/figures/`

That makes the demo lightweight and reproducible for grading. Full raw-data reproduction instructions are in the repository `README.md`.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

REPO_URL = 'https://github.com/ishaantibdewal/esg-news-risk-forecaster.git'
REPO_DIRNAME = 'esg-news-risk-forecaster'


def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def find_repo_root() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path('/content') / REPO_DIRNAME,
    ]
    for candidate in candidates:
        if (candidate / 'outputs' / 'tables').exists() and (candidate / 'report' / 'figures').exists():
            return candidate.resolve()

    if running_in_colab():
        target = Path('/content') / REPO_DIRNAME
        if not target.exists():
            subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(target)], check=True)
        return target.resolve()

    raise FileNotFoundError(
        'Could not find repository artifacts. Run this notebook from the repository root, '
        'or open it in Colab after the GitHub repository has been pushed.'
    )


ROOT = find_repo_root()
OUTPUTS = ROOT / 'outputs'
TABLES = OUTPUTS / 'tables'
OUTPUT_FIGURES = OUTPUTS / 'figures'
REPORT_FIGURES = ROOT / 'report' / 'figures'

pd.set_option('display.max_columns', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
plt.style.use('default')

print(f'Using repository root: {ROOT}')
print(f'Tables directory exists: {TABLES.exists()}')
print(f'Report figures directory exists: {REPORT_FIGURES.exists()}')

In [ ]:
required_files = [
    TABLES / 'raw_data_inventory.csv',
    TABLES / 'news_ticker_coverage.csv',
    TABLES / 'news_coverage_by_ticker_year.csv',
    TABLES / 'target_distribution.csv',
    TABLES / 'tuned_model_metrics.csv',
    TABLES / 'tuned_vs_untuned.csv',
    TABLES / 'predicted_risk_quintiles.csv',
    TABLES / 'tuned_test_predictions.csv',
    REPORT_FIGURES / 'dataset_distribution.png',
    REPORT_FIGURES / 'model_performance_regression.png',
    REPORT_FIGURES / 'roc_curve.png',
]

missing = [str(path.relative_to(ROOT)) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required generated artifacts:\n' + '\n'.join(missing))


def read_table(name: str, **kwargs) -> pd.DataFrame:
    return pd.read_csv(TABLES / name, **kwargs)


def show_fig(path: Path, width: int = 850) -> None:
    if not path.exists():
        raise FileNotFoundError(path)
    display(Image(filename=str(path), width=width))


raw_inventory = read_table('raw_data_inventory.csv')
news_ticker_coverage = read_table('news_ticker_coverage.csv')
news_coverage_by_year = read_table('news_coverage_by_ticker_year.csv')
universe_coverage = read_table('universe_coverage.csv')
target_distribution = read_table('target_distribution.csv')
panel_missingness = read_table('panel_missingness.csv')
tuned_metrics = read_table('tuned_model_metrics.csv')
tuned_vs_untuned = read_table('tuned_vs_untuned.csv')
risk_quintiles = read_table('predicted_risk_quintiles.csv')
predictions = read_table('tuned_test_predictions.csv', parse_dates=['week_end_date'])

print('Loaded tables:')
for name, frame in {
    'raw_inventory': raw_inventory,
    'news_ticker_coverage': news_ticker_coverage,
    'news_coverage_by_year': news_coverage_by_year,
    'universe_coverage': universe_coverage,
    'target_distribution': target_distribution,
    'tuned_metrics': tuned_metrics,
    'predictions': predictions,
}.items():
    print(f'  {name:<24} {frame.shape}')

## 1. Project Criteria Checklist

The project instructions ask for five components: dataset, predictive task, model, literature, and results. This notebook focuses on the empirical evidence for those components; the final PDF report contains the full prose and citations.

In [ ]:
criteria = pd.DataFrame({
    'Project component': [
        'Dataset and EDA',
        'Predictive task',
        'Model',
        'Literature',
        'Results',
        'Reproducibility',
    ],
    'Where it appears': [
        'FNSPID news + price histories; coverage, target, and text diagnostics below',
        '21-trading-day realized volatility regression and high-volatility classification',
        'Price baselines, text-only models, combined price+news models, tuned tree ensembles',
        'Discussed in report/main.pdf: ESG2Risk, FNSPID, FinBERT, ClimateBERT, text-risk literature',
        'Model comparisons, ablations, tuned-vs-untuned checks, quintile analysis, diagnostics',
        'README gives setup, data layout, staged pipeline commands, Colab notebook usage, and report build instructions',
    ],
})
criteria

## 2. Dataset Summary

The raw data source is FNSPID, which contains a large financial-news CSV and ticker-level price histories. The modeling dataset is a weekly panel: one row is one ticker-week, with past market features, same-week aggregated news features, and a future risk label.

In [ ]:
raw_news_gb = float(raw_inventory.loc[raw_inventory['item'].eq('news_csv_gb'), 'value'].iloc[0])
price_file_count = int(raw_inventory.loc[raw_inventory['item'].eq('price_file_count'), 'value'].iloc[0])
filtered_news_rows = int(news_ticker_coverage['news_rows'].sum())
modeled_article_count = int(news_coverage_by_year['article_count'].sum())
model_rows = len(predictions)
modeled_tickers = predictions['ticker'].nunique()
news_weeks = int(predictions['has_esg_news_week'].sum())
scored_test_rows = int(predictions.filter(like='tuned_reg__').notna().any(axis=1).sum())

summary = pd.DataFrame({
    'Quantity': [
        'Raw FNSPID news file size (GB)',
        'Ticker price-history files available',
        'Filtered ticker-news rows',
        'Modeled-year article counts in diagnostics',
        'Final supervised ML rows (ticker-weeks)',
        'Modeled tickers',
        'Ticker-weeks with ESG/news signals',
        'Held-out 2023 rows with saved tuned predictions',
        'Panel start',
        'Panel end',
    ],
    'Value': [
        f'{raw_news_gb:,.2f}',
        f'{price_file_count:,}',
        f'{filtered_news_rows:,}',
        f'{modeled_article_count:,}',
        f'{model_rows:,}',
        f'{modeled_tickers:,}',
        f'{news_weeks:,}',
        f'{scored_test_rows:,}',
        predictions['week_end_date'].min().date(),
        predictions['week_end_date'].max().date(),
    ],
})
summary

**Dataset takeaway:** the raw news corpus is large, but the supervised learning problem is a structured weekly panel. That is why chronological validation, careful baselines, and ranking diagnostics matter more than simply reporting a large raw file size.

## 3. Exploratory Data Analysis

The EDA checks whether the dataset is suitable for the prediction task and where the likely modeling challenges are: uneven coverage, sparse news signals, noisy keyword filtering, and a right-skewed volatility target.

In [ ]:
display(Markdown('### News coverage and ESG categories'))
show_fig(REPORT_FIGURES / 'dataset_distribution.png', width=900)
show_fig(REPORT_FIGURES / 'esg_category_distribution.png', width=720)

coverage_top = news_ticker_coverage.head(10).copy()
display(Markdown('### Top tickers by filtered news rows'))
display(coverage_top)

In [ ]:
display(Markdown('### Text-signal and target distributions'))
show_fig(REPORT_FIGURES / 'sentiment_distribution.png', width=720)
show_fig(REPORT_FIGURES / 'risk_label_distribution.png', width=720)

display(Markdown('### Future volatility target summary'))
display(target_distribution)

**EDA interpretation:** news coverage is not uniform across firms and years, so text features need to be evaluated separately on all weeks and on news-covered weeks. The volatility target is right-skewed, so the project uses both regression and high-risk classification/ranking diagnostics.

## 4. Modeling Setup

The supervised input unit is a ticker-week observation. The model sees past-only market features and same-week aggregated news features, then predicts future risk.

Chronological split:

- **Train:** 2016-2021
- **Validation:** 2022
- **Test:** 2023

Targets:

- **Regression:** future 21-trading-day realized volatility.
- **Classification:** top-quartile high-volatility ticker-weeks within each week.

Feature groups:

- Price-only baseline.
- ESG keyword features.
- FinBERT financial sentiment.
- ClimateBERT climate relevance.
- Combined price + text feature groups.

In [ ]:
split_counts = pd.Series({
    'train_2016_2021': int(((predictions['week_end_date'] >= '2016-01-01') & (predictions['week_end_date'] <= '2021-12-31')).sum()),
    'validation_2022': int(((predictions['week_end_date'] >= '2022-01-01') & (predictions['week_end_date'] <= '2022-12-31')).sum()),
    'test_2023': int((predictions['week_end_date'] >= '2023-01-01').sum()),
    'test_2023_news_weeks': int(((predictions['week_end_date'] >= '2023-01-01') & predictions['has_esg_news_week']).sum()),
}).rename('rows').to_frame()
split_counts

## 5. Main Tuned Results

The table below selects the best tuned model within each feature group on the 2023 test period. Lower RMSE is better for regression; higher ROC-AUC is better for classification.

In [ ]:
reg = tuned_metrics.query("evaluation_slice == 'all_weeks' and task == 'regression'").copy()
clf = tuned_metrics.query("evaluation_slice == 'all_weeks' and task == 'classification'").copy()

best_reg = reg.loc[reg.groupby('feature_group')['rmse'].idxmin()].sort_values('rmse')
best_clf = clf.loc[clf.groupby('feature_group')['roc_auc'].idxmax()].sort_values('roc_auc', ascending=False)

best_reg_view = best_reg[['feature_group', 'model', 'test_rows', 'rmse', 'mae', 'r2', 'spearman']].reset_index(drop=True)
best_clf_view = best_clf[['feature_group', 'model', 'test_rows', 'roc_auc', 'pr_auc', 'f1', 'precision_at_top_10pct']].reset_index(drop=True)

display(Markdown('### Best regression model by feature group'))
display(best_reg_view)

display(Markdown('### Best classification model by feature group'))
display(best_clf_view)

In [ ]:
price_reg = best_reg.set_index('feature_group').loc['price']
climate_reg = best_reg.set_index('feature_group').loc['price_climatebert']
price_clf = best_clf.set_index('feature_group').loc['price']
climate_clf = best_clf.set_index('feature_group').loc['price_climatebert']

main_result = pd.DataFrame({
    'Task': ['Regression RMSE', 'Classification ROC-AUC'],
    'Price-only baseline': [price_reg['rmse'], price_clf['roc_auc']],
    'Price + ClimateBERT': [climate_reg['rmse'], climate_clf['roc_auc']],
    'Difference': [price_reg['rmse'] - climate_reg['rmse'], climate_clf['roc_auc'] - price_clf['roc_auc']],
})
main_result

**Main result:** the tuned **price + ClimateBERT** model is the best selected configuration in both tasks. The improvement over price-only is modest, so the correct conclusion is incremental value rather than a dramatic replacement of market features.

In [ ]:
show_fig(REPORT_FIGURES / 'model_performance_regression.png', width=900)
show_fig(REPORT_FIGURES / 'model_performance_classification.png', width=900)

## 6. Ablation and Hyperparameter Checks

A good result should not only report the best number. It should also ask whether the feature groups help, whether all text features are useful, and whether tuning changes the conclusion.

In [ ]:
groups = ['price', 'price_climatebert', 'price_all_news', 'full']
reg_best = best_reg.set_index('feature_group').loc[groups]
clf_best = best_clf.set_index('feature_group').loc[groups]

ablation = pd.DataFrame({
    'rmse': reg_best['rmse'],
    'delta_rmse_vs_price': reg_best.loc['price', 'rmse'] - reg_best['rmse'],
    'roc_auc': clf_best['roc_auc'],
    'delta_auc_vs_price': clf_best['roc_auc'] - clf_best.loc['price', 'roc_auc'],
}).reset_index(names='feature_group')

ablation

In [ ]:
display(Markdown('### Tuned vs. untuned comparison'))
display(tuned_vs_untuned)

show_fig(REPORT_FIGURES / 'ablation_regression_delta.png', width=760)
show_fig(REPORT_FIGURES / 'ablation_classification_delta.png', width=760)

**Ablation takeaway:** adding all text features is not automatically better. The more targeted climate-aware feature group is the cleanest combined result, while broader text features can add noise as well as signal.

## 7. Classification Diagnostics

For risk screening, ranking quality is often more important than a fixed threshold. ROC-AUC and precision-at-top-risk help answer whether the model can prioritize ticker-weeks that deserve attention.

In [ ]:
show_fig(REPORT_FIGURES / 'roc_curve.png', width=680)
show_fig(REPORT_FIGURES / 'confusion_matrix.png', width=560)

clf_cols = ['feature_group', 'model', 'roc_auc', 'pr_auc', 'f1', 'precision_at_top_10pct', 'tn', 'fp', 'fn', 'tp']
display(best_clf.query("feature_group in ['price', 'price_climatebert', 'price_all_news', 'full']")[clf_cols])

**Classification interpretation:** the selected model separates high-risk ticker-weeks well, but errors remain near the threshold. It is best framed as an analyst screening tool, not an automatic decision rule.

## 8. Regression and Risk-Ranking Diagnostics

The regression model is not just evaluated by point error. The quintile test checks whether observations assigned higher predicted risk actually experience higher realized volatility.

In [ ]:
show_fig(REPORT_FIGURES / 'predicted_vs_realized_risk.png', width=680)
show_fig(REPORT_FIGURES / 'quintile_realized_volatility.png', width=760)
show_fig(REPORT_FIGURES / 'timeseries_predicted_realized_risk.png', width=900)

display(Markdown('### ESG2Risk-style predicted-risk quintiles'))
display(risk_quintiles)

**Ranking interpretation:** realized volatility generally rises across predicted-risk quintiles. This supports the use case as a risk-ranking model even when exact realized volatility remains noisy.

## 9. Feature and Error Diagnostics

The final diagnostic asks whether the combined model is still grounded in market risk while using text as a secondary signal.

In [ ]:
show_fig(REPORT_FIGURES / 'feature_importance.png', width=820)

missing_focus = panel_missingness.head(12).copy()
display(Markdown('### Highest missingness features in the weekly panel'))
display(missing_focus)

The feature association plot is a transparent correlation-style diagnostic, not a SHAP plot. The important pattern is that market-risk features remain central, while climate-aware text contributes as additional context.

## 10. Inspect Saved Predictions

This section makes the demo concrete: choose a saved model column, rank the 2023 test ticker-weeks by predicted score, and compare predictions with realized outcomes.

In [ ]:
best_reg_col = 'tuned_reg__price_climatebert__hist_gradient_boosting'
best_clf_col = 'tuned_clf__price_climatebert__hist_gradient_boosting'

scored_2023 = predictions.loc[predictions['week_end_date'] >= '2023-01-01'].copy()
scored_2023 = scored_2023.dropna(subset=[best_reg_col, best_clf_col], how='all')

high_risk_examples = (
    scored_2023[['ticker', 'week_end_date', 'has_esg_news_week', 'future_vol_21d', 'high_vol_21d', best_reg_col, best_clf_col]]
    .sort_values(best_clf_col, ascending=False)
    .head(12)
    .rename(columns={
        best_reg_col: 'predicted_volatility',
        best_clf_col: 'predicted_high_vol_probability',
    })
)

high_risk_examples

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
plot_data = scored_2023.dropna(subset=[best_reg_col, 'future_vol_21d']).copy()
ax.scatter(plot_data[best_reg_col], plot_data['future_vol_21d'], alpha=0.55, s=28)
lo = min(plot_data[best_reg_col].min(), plot_data['future_vol_21d'].min())
hi = max(plot_data[best_reg_col].max(), plot_data['future_vol_21d'].max())
ax.plot([lo, hi], [lo, hi], linestyle='--', linewidth=1, color='black')
ax.set_title('Saved 2023 predictions: price + ClimateBERT HGB')
ax.set_xlabel('Predicted 21d realized volatility')
ax.set_ylabel('Actual future 21d realized volatility')
ax.grid(alpha=0.25)
fig.tight_layout()
display(fig)
plt.close(fig)

## 11. Final Interpretation for the Grader

The project supports a cautious, defensible conclusion:

- Price-history features are hard to beat for volatility forecasting.
- News-only models are not competitive with market-only baselines.
- ESG keyword and general sentiment features are informative but noisy.
- ClimateBERT adds the clearest incremental value when combined with price features.
- The model is best understood as a risk-ranking and analyst-screening aid.
- The evidence is predictive, not causal.

The practical takeaway is: **ESG news signals enrich a price-based risk model, but should not replace market-risk features.**

## 12. Reproducibility Notes

This notebook demonstrates generated results. To duplicate the full pipeline from raw data, follow the top-level `README.md`:

1. Place FNSPID under `data/raw/fnspid/`.
2. Install `requirements.txt`.
3. Run the staged `python -m src.pipeline ...` commands.
4. Regenerate outputs, report tables, figures, and notebook/report artifacts.

The raw data and intermediate Parquet files are intentionally excluded from GitHub because they are too large for a normal submission repository.